# Supplementary Figure S2 — model response traces

Run cells in order. Calculated panels are saved under `figures/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotting as p
from stim_gen import makeCHIRPstim, makePPFstim

import jax.numpy as jnp

def nmax(x):
    return x / np.nanmax(x)

## Compute or load the AN1 traces

In [ ]:
RUN_EXPENSIVE = False
CACHE = (ROOT / "results" if RUN_EXPENSIVE else SAVED_RESULTS) / "figS2_traces.npz"

if RUN_EXPENSIVE:
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    input_stim, _, _, _ = makeCHIRPstim(
        cdur=np.array([120]), cpau=np.array([120]),
        ppau=20, pdur=20, Fs=1000, stim_len=50_000,
    )
    input_stim = jnp.array(input_stim)
    from model.model_20210125_params import Params
    from model.model_20210125_structure import run_model
    from model.model_20210125_params_fix import Params as Params_fix
    from model.model_20210125_structure_fix import run_model as run_model_fix

    old = run_model(input_stim, Params(), verbose_output=True, return_dict=True)["an1"]
    modified = run_model_fix(input_stim, Params_fix(), verbose_output=True, return_dict=True)["an1"]
    offsets = range(0, 1500, 240)
    old_traces = np.array([old[50+offset:200+offset, 0] for offset in offsets])
    fix_traces = np.array([modified[50+offset:200+offset, 0] for offset in offsets])
    np.savez_compressed(CACHE, old=old_traces, modified=fix_traces)
else:
    with np.load(CACHE) as cached:
        old_traces = cached["old"]
        fix_traces = cached["modified"]

## Plot seven successive chirps

In [ ]:
N = len(old_traces)
colors = plt.get_cmap("Reds")(np.linspace(.3, 1, N))[::-1]
fig, axes = plt.subplots(2, 2, figsize=(7, 5), sharex='row')
for column, (traces, title) in enumerate([
    (old_traces, 'Original model'), (fix_traces, 'Modified model')
]):
    for index in [0, 3, 6]:
        axes[0, column].plot(traces[index], color=colors[index],
                             label=f'Chirp {index + 1}')
    axes[0, column].set(xlim=(0, 150), ylim=(0, 5), title=title,
                        xlabel='Time [ms]')
    response = traces[:, 100:110].mean(axis=1)
    axes[1, column].plot(np.arange(1, N + 1), response, color='black')
    for index, value in enumerate(response):
        axes[1, column].plot(index + 1, value, 'o', color=colors[index])
    axes[1, column].set(xlabel='Chirp number', xticks=range(1, N + 1))
axes[0, 0].set_ylabel('AN1 response')
axes[1, 0].set_ylabel('AN1 response to last pulse')
axes[0, 1].legend(frameon=False, handlelength=0, labelcolor='linecolor')
fig.tight_layout()
p.label_axes(fig, labels=['A', 'C', 'B', 'D'], size=14, loc=(.02, .95))
fig.savefig(FIGURES / 'fig_S2.pdf', bbox_inches='tight')
plt.show()